In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
from PIL import Image
from dataset import loading_data
import constants
from train import train_model
from eval import evaluate_model

### Loading Data

In [2]:
image_datasets = loading_data.get_customdataset()
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=1,
                                            shuffle=True if x=="train" else False, num_workers=4)
            for x in image_datasets}
dataset_sizes = {x: len(image_datasets[x]) for x in image_datasets}
class_names = image_datasets['train'].classes

In [3]:
inputs, classes = next(iter(dataloaders['train']))

### Loading Model

In [4]:
model_ft = models.resnet18(weights='IMAGENET1K_V1')
num_ftrs = model_ft.fc.in_features

model_ft.fc = nn.Linear(num_ftrs, len(class_names))

model_ft = model_ft.to(constants.DEVICE)

criterion = nn.CrossEntropyLoss()

# Observe that all parameters are being optimized
optimizer_ft = optim.SGD(model_ft.parameters(), lr=0.001, momentum=0.9)

# Decay LR by a factor of 0.1 every 7 epochs
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)

### Training and Evaluation

In [5]:
model_ft = train_model(model_ft, dataloaders, dataset_sizes, criterion, optimizer_ft, exp_lr_scheduler, num_epochs=2)

Epoch 0/1
----------


train Loss: 0.0014 Acc: 0.0000
val Loss: 0.0104 Acc: 0.0000
Epoch 1/1
----------
train Loss: 0.0014 Acc: 0.0000
val Loss: 0.0105 Acc: 0.0000
Training complete in 0m 35s
Best val Acc: 0.000000


In [28]:
model_ft = models.resnet18()
num_ftrs = model_ft.fc.in_features

model_ft.fc = nn.Linear(num_ftrs, len(class_names))

model_ft = model_ft.to(constants.DEVICE)
best_model_params_path = os.path.join(constants.CKPT_PATH, f"ckpt_0.pt")
model_ft.load_state_dict(torch.load(best_model_params_path, weights_only=True))

<All keys matched successfully>

In [7]:
evaluate_model(model_ft, dataloaders["test"])

Accuracy per class:
Forell: 0.0000
Förgylld braxen: 0.0000
Havsaborre: 0.0000
Makrill: 0.0000
Randig multe: 0.0000
Räka: 0.0000
Röd braxen: 0.0000
Röd multe: 0.0000
Skarpsill: 0.0000

Overall Accuracy: 0.0000


In [29]:
model_ft.eval()
print("eval called")

eval called


In [4]:
from dataset.loading_data import DATA_TRANSFORM
from PIL import Image

In [64]:
img_path = "/Users/hash/hamza_data/Work/github/fish-classification/data/split_data/test/Forell/00013.png"
img = Image.open(img_path)


In [63]:
img = img.unsqueeze(0)
img.size

AttributeError: 'PngImageFile' object has no attribute 'unsqueeze'

In [56]:
img = DATA_TRANSFORM['test'](img)

In [34]:
img = img.unsqueeze(0)
img = img.to(constants.DEVICE)

In [35]:
img.shape

torch.Size([1, 3, 224, 224])

In [36]:
out = model_ft(img)

In [37]:
out

tensor([[-0.4545,  0.1588,  0.0840, -0.5309,  0.0882, -0.1877,  0.3620,  0.0414,
         -0.1511]], grad_fn=<AddmmBackward0>)